In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import DeltaTable

In [0]:
# key_col_list = eval("['flight_id']")
# catalog = 'dbx_dbt_flight'
# cdc_col = 'modified_date'
# backdated_refresh = ''
# source_object = 'flights'
# source_schema = 'silver'
# target_schema = 'gold'
# target_object = 'flights'
# surrogate_key = 'dim_flights_key'

In [0]:
# key_col_list = eval("['airport_id']")
# catalog = 'dbx_dbt_flight'
# cdc_col = 'modified_date'
# backdated_refresh = ''
# source_object = 'airports'
# source_schema = 'silver'
# target_schema = 'gold'
# target_object = 'airports'
# surrogate_key = 'dim_airports_key'

In [0]:
key_col_list = eval("['passenger_id']")
catalog = 'dbx_dbt_flight'
cdc_col = 'modified_date'
backdated_refresh = ''
source_object = 'passengers'
source_schema = 'silver'
target_schema = 'gold'
target_object = 'passengers'
surrogate_key = 'dim_passengers_key'

In [0]:
if len(backdated_refresh) == 0:
    if spark.catalog.tableExists(f"{catalog}.{target_schema}.{target_object}"):
        last_load_date = spark.sql(f"SELECT MAX({cdc_col}) FROM dbx_dbt_flight.{target_schema}.{target_object}").collect()[0][0]

    else:
        last_load_date = "1900-01-01 00:00:00"

else:
    last_load_date = backdated_refresh

In [0]:
last_load_date

'1900-01-01 00:00:00'

In [0]:
df_source = spark.sql(f"SELECT * FROM {catalog}.{source_schema}.{source_object} WHERE {cdc_col} >= '{last_load_date}'")

In [0]:
key_col_list

['passenger_id']

In [0]:
key_col_string_inc = ', '.join(key_col_list)
key_col_string_inc

'passenger_id'

In [0]:
key_col_string_init = [f"'' AS {i}" for i in key_col_list]
key_col_string_init = ', '.join(key_col_string_init)
key_col_string_init

"'' AS passenger_id"

In [0]:
if spark.catalog.tableExists(f"{catalog}.{target_schema}.{target_object}"):
    #incremental load

    key_col_string_inc = ', '.join(key_col_list)

    df_target = spark.sql(f"SELECT {key_col_string_inc}, {surrogate_key}, create_date, update_date FROM {catalog}.{target_schema}.{target_object}")
else:
    #initial load
    key_col_string_init = [f"'' AS {i}" for i in key_col_list]
    key_col_string_init = ', '.join(key_col_string_init)

    #helps to create a pseudo table
    df_target = spark.sql(f"SELECT {key_col_string_init}, CAST('0' AS INT) AS {surrogate_key}, CAST('1900-01-01 00:00:00' AS TIMESTAMP) AS create_date, CAST ('1900-01-01 00:00:00' AS TIMESTAMP) AS update_date  WHERE 1 = 0")

In [0]:
spark.sql(f"SELECT {key_col_string_inc} FROM {catalog}.{source_schema}.{source_object}")

DataFrame[passenger_id: string]

In [0]:
df_target.display()

passenger_id,dim_passengers_key,create_date,update_date


In [0]:
join_condition = ' AND '.join([f"src.{i} = trg.{i}" for i in key_col_list])

In [0]:
df_source.createOrReplaceTempView('src')
df_target.createOrReplaceTempView('trg')

df_join = spark.sql(f"""
             SELECT src.*,
                    trg.{surrogate_key},
                    trg.create_date,
                    trg.update_date
            FROM src 
            LEFT JOIN trg
            ON {join_condition}   
             """)

In [0]:
df_join.display()

passenger_id,name,gender,nationality,modified_date,dim_passengers_key,create_date,update_date
P0001,Kevin Ferguson,Male,Reunion,2026-06-10T10:13:29.169Z,null,null,null
P0002,Kathleen Martinez DVM,Female,Burkina Faso,2026-06-10T10:13:29.169Z,null,null,null
P0003,Cynthia Frazier,Male,Marshall Islands,2026-06-10T10:13:29.169Z,null,null,null
P0004,Ryan Ramsey,Male,Niger,2026-06-10T10:13:29.169Z,null,null,null
P0005,Mike Kim,Male,Taiwan,2026-06-10T10:13:29.169Z,null,null,null
P0006,Diana Adams,Male,Mayotte,2026-06-10T10:13:29.169Z,null,null,null
P0007,Sharon Moon,Male,Madagascar,2026-06-10T10:13:29.169Z,null,null,null
P0008,Cheryl Glenn,Male,Maldives,2026-06-10T10:13:29.169Z,null,null,null
P0009,Allen Lowery,Male,Rwanda,2026-06-10T10:13:29.169Z,null,null,null
P0010,Maria Medina,Male,Denmark,2026-06-10T10:13:29.169Z,null,null,null


In [0]:
df_old = df_join.filter(col(f'{surrogate_key}').isNotNull()) #old records
df_new = df_join.filter(col(f'{surrogate_key}').isNull())   #new records

In [0]:
df_old_enr = df_old.withColumn('update_date', current_timestamp())
df_old_enr.display()

passenger_id,name,gender,nationality,modified_date,dim_passengers_key,create_date,update_date


In [0]:
df_new.display()

passenger_id,name,gender,nationality,modified_date,dim_passengers_key,create_date,update_date
P0001,Kevin Ferguson,Male,Reunion,2026-06-10T10:13:29.169Z,null,null,null
P0002,Kathleen Martinez DVM,Female,Burkina Faso,2026-06-10T10:13:29.169Z,null,null,null
P0003,Cynthia Frazier,Male,Marshall Islands,2026-06-10T10:13:29.169Z,null,null,null
P0004,Ryan Ramsey,Male,Niger,2026-06-10T10:13:29.169Z,null,null,null
P0005,Mike Kim,Male,Taiwan,2026-06-10T10:13:29.169Z,null,null,null
P0006,Diana Adams,Male,Mayotte,2026-06-10T10:13:29.169Z,null,null,null
P0007,Sharon Moon,Male,Madagascar,2026-06-10T10:13:29.169Z,null,null,null
P0008,Cheryl Glenn,Male,Maldives,2026-06-10T10:13:29.169Z,null,null,null
P0009,Allen Lowery,Male,Rwanda,2026-06-10T10:13:29.169Z,null,null,null
P0010,Maria Medina,Male,Denmark,2026-06-10T10:13:29.169Z,null,null,null


In [0]:
if spark.catalog.tableExists(f"{catalog}.{target_schema}.{target_object}"):
    max_surrogate_key = spark.sql(f'SELECT MAX({surrogate_key}) FROM {catalog}.{target_schema}.{target_object}').collect()[0][0]
    df_new_enr = df_new.withColumn(f'{surrogate_key}', (monotonically_increasing_id() + lit(max_surrogate_key) + lit(1)))\
                    .withColumn('create_date', current_timestamp())\
                    .withColumn('update_date', current_timestamp())


else:
    max_surrogate_key = 0
    df_new_enr = df_new.withColumn(f'{surrogate_key}', (monotonically_increasing_id() + lit(max_surrogate_key) + lit(1)))\
                    .withColumn('create_date', current_timestamp())\
                    .withColumn('update_date', current_timestamp())

In [0]:
spark.sql(f'SELECT MAX({key_col_string_inc}) FROM {catalog}.{source_schema}.{source_object}').collect()[0][0]

'P0225'

In [0]:
key_col_string_inc


'passenger_id'

In [0]:
df_new_enr.display()

passenger_id,name,gender,nationality,modified_date,dim_passengers_key,create_date,update_date
P0001,Kevin Ferguson,Male,Reunion,2026-06-10T10:13:29.169Z,1,2026-06-16T11:13:36.272Z,2026-06-16T11:13:36.272Z
P0002,Kathleen Martinez DVM,Female,Burkina Faso,2026-06-10T10:13:29.169Z,2,2026-06-16T11:13:36.272Z,2026-06-16T11:13:36.272Z
P0003,Cynthia Frazier,Male,Marshall Islands,2026-06-10T10:13:29.169Z,3,2026-06-16T11:13:36.272Z,2026-06-16T11:13:36.272Z
P0004,Ryan Ramsey,Male,Niger,2026-06-10T10:13:29.169Z,4,2026-06-16T11:13:36.272Z,2026-06-16T11:13:36.272Z
P0005,Mike Kim,Male,Taiwan,2026-06-10T10:13:29.169Z,5,2026-06-16T11:13:36.272Z,2026-06-16T11:13:36.272Z
P0006,Diana Adams,Male,Mayotte,2026-06-10T10:13:29.169Z,6,2026-06-16T11:13:36.272Z,2026-06-16T11:13:36.272Z
P0007,Sharon Moon,Male,Madagascar,2026-06-10T10:13:29.169Z,7,2026-06-16T11:13:36.272Z,2026-06-16T11:13:36.272Z
P0008,Cheryl Glenn,Male,Maldives,2026-06-10T10:13:29.169Z,8,2026-06-16T11:13:36.272Z,2026-06-16T11:13:36.272Z
P0009,Allen Lowery,Male,Rwanda,2026-06-10T10:13:29.169Z,9,2026-06-16T11:13:36.272Z,2026-06-16T11:13:36.272Z
P0010,Maria Medina,Male,Denmark,2026-06-10T10:13:29.169Z,10,2026-06-16T11:13:36.272Z,2026-06-16T11:13:36.272Z


In [0]:
df_old_enr.display()

passenger_id,name,gender,nationality,modified_date,dim_passengers_key,create_date,update_date


In [0]:
df_union = df_old_enr.unionByName(df_new_enr)
df_union.display()


passenger_id,name,gender,nationality,modified_date,dim_passengers_key,create_date,update_date
P0001,Kevin Ferguson,Male,Reunion,2026-06-10T10:13:29.169Z,1,2026-06-16T11:13:37.651Z,2026-06-16T11:13:37.651Z
P0002,Kathleen Martinez DVM,Female,Burkina Faso,2026-06-10T10:13:29.169Z,2,2026-06-16T11:13:37.651Z,2026-06-16T11:13:37.651Z
P0003,Cynthia Frazier,Male,Marshall Islands,2026-06-10T10:13:29.169Z,3,2026-06-16T11:13:37.651Z,2026-06-16T11:13:37.651Z
P0004,Ryan Ramsey,Male,Niger,2026-06-10T10:13:29.169Z,4,2026-06-16T11:13:37.651Z,2026-06-16T11:13:37.651Z
P0005,Mike Kim,Male,Taiwan,2026-06-10T10:13:29.169Z,5,2026-06-16T11:13:37.651Z,2026-06-16T11:13:37.651Z
P0006,Diana Adams,Male,Mayotte,2026-06-10T10:13:29.169Z,6,2026-06-16T11:13:37.651Z,2026-06-16T11:13:37.651Z
P0007,Sharon Moon,Male,Madagascar,2026-06-10T10:13:29.169Z,7,2026-06-16T11:13:37.651Z,2026-06-16T11:13:37.651Z
P0008,Cheryl Glenn,Male,Maldives,2026-06-10T10:13:29.169Z,8,2026-06-16T11:13:37.651Z,2026-06-16T11:13:37.651Z
P0009,Allen Lowery,Male,Rwanda,2026-06-10T10:13:29.169Z,9,2026-06-16T11:13:37.651Z,2026-06-16T11:13:37.651Z
P0010,Maria Medina,Male,Denmark,2026-06-10T10:13:29.169Z,10,2026-06-16T11:13:37.651Z,2026-06-16T11:13:37.651Z


In [0]:
df_old_enr.display()
df_new_enr.display()

passenger_id,name,gender,nationality,modified_date,dim_passengers_key,create_date,update_date


passenger_id,name,gender,nationality,modified_date,dim_passengers_key,create_date,update_date
P0001,Kevin Ferguson,Male,Reunion,2026-06-10T10:13:29.169Z,1,2026-06-16T11:13:38.803Z,2026-06-16T11:13:38.803Z
P0002,Kathleen Martinez DVM,Female,Burkina Faso,2026-06-10T10:13:29.169Z,2,2026-06-16T11:13:38.803Z,2026-06-16T11:13:38.803Z
P0003,Cynthia Frazier,Male,Marshall Islands,2026-06-10T10:13:29.169Z,3,2026-06-16T11:13:38.803Z,2026-06-16T11:13:38.803Z
P0004,Ryan Ramsey,Male,Niger,2026-06-10T10:13:29.169Z,4,2026-06-16T11:13:38.803Z,2026-06-16T11:13:38.803Z
P0005,Mike Kim,Male,Taiwan,2026-06-10T10:13:29.169Z,5,2026-06-16T11:13:38.803Z,2026-06-16T11:13:38.803Z
P0006,Diana Adams,Male,Mayotte,2026-06-10T10:13:29.169Z,6,2026-06-16T11:13:38.803Z,2026-06-16T11:13:38.803Z
P0007,Sharon Moon,Male,Madagascar,2026-06-10T10:13:29.169Z,7,2026-06-16T11:13:38.803Z,2026-06-16T11:13:38.803Z
P0008,Cheryl Glenn,Male,Maldives,2026-06-10T10:13:29.169Z,8,2026-06-16T11:13:38.803Z,2026-06-16T11:13:38.803Z
P0009,Allen Lowery,Male,Rwanda,2026-06-10T10:13:29.169Z,9,2026-06-16T11:13:38.803Z,2026-06-16T11:13:38.803Z
P0010,Maria Medina,Male,Denmark,2026-06-10T10:13:29.169Z,10,2026-06-16T11:13:38.803Z,2026-06-16T11:13:38.803Z


In [0]:
if spark.catalog.tableExists(f"{catalog}.{target_schema}.{target_object}"):
    delta_obj = DeltaTable.forName(spark, f"{catalog}.{target_schema}.{target_object}")
    delta_obj.alias("trg").merge(df_union.alias("src"),  f'trg.{surrogate_key} = src.{surrogate_key}')\
                    .whenMatchedUpdateAll(condition = f"src.{cdc_col} >= trg.{cdc_col}")\
                    .whenNotMatchedInsertAll()\
                    .execute()

else:
    df_union.write.format('delta')\
                .mode('append')\
                .saveAsTable(f"{catalog}.{target_schema}.{target_object}") 

In [0]:
%sql
SELECT * FROM dbx_dbt_flight.gold.passengers

passenger_id,name,gender,nationality,modified_date,dim_passengers_key,create_date,update_date
P0001,Kevin Ferguson,Male,Reunion,2026-06-10T10:13:29.169Z,1,2026-06-16T11:13:42.335Z,2026-06-16T11:13:42.335Z
P0002,Kathleen Martinez DVM,Female,Burkina Faso,2026-06-10T10:13:29.169Z,2,2026-06-16T11:13:42.335Z,2026-06-16T11:13:42.335Z
P0003,Cynthia Frazier,Male,Marshall Islands,2026-06-10T10:13:29.169Z,3,2026-06-16T11:13:42.335Z,2026-06-16T11:13:42.335Z
P0004,Ryan Ramsey,Male,Niger,2026-06-10T10:13:29.169Z,4,2026-06-16T11:13:42.335Z,2026-06-16T11:13:42.335Z
P0005,Mike Kim,Male,Taiwan,2026-06-10T10:13:29.169Z,5,2026-06-16T11:13:42.335Z,2026-06-16T11:13:42.335Z
P0006,Diana Adams,Male,Mayotte,2026-06-10T10:13:29.169Z,6,2026-06-16T11:13:42.335Z,2026-06-16T11:13:42.335Z
P0007,Sharon Moon,Male,Madagascar,2026-06-10T10:13:29.169Z,7,2026-06-16T11:13:42.335Z,2026-06-16T11:13:42.335Z
P0008,Cheryl Glenn,Male,Maldives,2026-06-10T10:13:29.169Z,8,2026-06-16T11:13:42.335Z,2026-06-16T11:13:42.335Z
P0009,Allen Lowery,Male,Rwanda,2026-06-10T10:13:29.169Z,9,2026-06-16T11:13:42.335Z,2026-06-16T11:13:42.335Z
P0010,Maria Medina,Male,Denmark,2026-06-10T10:13:29.169Z,10,2026-06-16T11:13:42.335Z,2026-06-16T11:13:42.335Z
